# WasteVision Training — upgraded pipeline (Colab, free GPU)

**Runtime > Change runtime type > GPU** (T4 is fine) before running.

This is an upgraded version of the baseline training notebook. The
original ran a single fixed-hyperparameter YOLOv8n training pass and
called it done. This version adds the things that actually move a
detector from "trains without crashing" to "has a defensible baseline
number and is ready to serve":

1. **Data-driven imbalance check** — measure the 8-class distribution
   before guessing whether it's a problem.
2. **Minority-class oversampling** (`src/data/rebalance.py`) — brings
   under-represented material groups up to a target share of train images.
3. **A real baseline run first** — small model, short schedule, logged to
   MLflow — so every later change has something concrete to beat, not a
   vibes-based "this seems better."
4. **Architecture + hyperparameter comparison** (n → s → m, tuned
   augmentation for small/cluttered litter objects) — all logged to the
   same MLflow experiment for a direct side-by-side.
5. **Confidence-threshold tuning** (`src/models/evaluate.py`) — the
   0.25 default is generic; this picks the threshold that actually
   maximizes F1 for *this* trained checkpoint.
6. **TTA-evaluated final metrics + full per-class report.**
7. **Export to ONNX + a latency benchmark** — sized for the CPU-tier
   Hugging Face Spaces deployment in `DEPLOYMENT.md`.
8. **A deployment sanity check** — runs the *exact* `InferenceService`
   class the FastAPI/Gradio apps use, against the exported weights,
   before Docker ever gets involved.

Every technique below is implemented in `src/`, not copy-pasted into this
notebook, and is covered by `tests/` (55 tests, no GPU needed) — this
notebook is a *driver* for tested code, not a place new untested logic
lives.

## 1. Setup

In [ ]:
!git clone https://github.com/<your-username>/wastevision.git
%cd wastevision
!pip install -q -r requirements.txt

import torch
print("GPU available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Download TACO
TACO's images are Flickr-hosted and fetched via TACO's own downloader — not redistributed directly. This can take a while and will skip any images that have since been deleted from Flickr (expect a small amount of unavoidable data loss).

In [ ]:
!git clone https://github.com/pedropro/TACO.git
%cd TACO
!python download.py
%cd ..

## 3. Convert to YOLO format (leak-free, stratified split)

In [ ]:
!python -m src.data.download_and_prepare \
    --images-dir TACO/data \
    --output-dir data/processed

## 4. Check the class distribution before guessing at fixes
Quantify the imbalance first — deciding to oversample (and by how much)
without looking at this is a guess, not an engineering decision.

In [ ]:
from pathlib import Path
from collections import Counter
from src.data.rebalance import dominant_group_from_label_file
from src.utils.taxonomy import MATERIAL_GROUPS

def split_distribution(processed_dir: Path, split: str) -> Counter:
    labels_dir = processed_dir / split / "labels"
    counts = Counter()
    for label_path in labels_dir.glob("*.txt"):
        group = dominant_group_from_label_file(label_path)
        if group is not None:
            counts[group] += 1
    return counts

processed_dir = Path("data/processed")
train_dist = split_distribution(processed_dir, "train")
print("Train images by dominant material group:")
for name in MATERIAL_GROUPS:
    print(f"  {name:16s} {train_dist.get(name, 0)}")

## 5. Oversample minority groups in the train split only
`min_fraction_of_majority=0.5` brings every group up to at least half the
majority group's image count. This only duplicates whole images (keeping
scene context intact) and only touches `train/` — `val`/`test` stay
untouched so evaluation numbers later aren't inflated by duplicates.
Re-running this cell is safe (idempotent).

In [ ]:
from src.data.rebalance import rebalance_train_split

result = rebalance_train_split(processed_dir, min_fraction_of_majority=0.5)
print(f"Added {result['added']} duplicated images.")
print("Before:", result["before"])
print("After: ", result["after"])

## 6. Baseline run
Small model, short schedule, default-ish hyperparameters, logged to
MLflow as `baseline_yolov8n`. This number is the floor everything below
has to beat — without it, "the improved run got 0.41 mAP50" has no
reference point.

In [ ]:
from src.models.train import train

baseline_config = {
    "data": str(processed_dir / "data.yaml"),
    "model": "yolov8n.pt",
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "patience": 15,
    "optimizer": "auto",
    "seed": 42,
    "device": 0,
    "workers": 8,
    # Default augmentation from configs/config.yaml — nothing tuned yet.
    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4, "degrees": 0.0,
    "translate": 0.1, "scale": 0.5, "fliplr": 0.5,
    "mosaic": 1.0, "close_mosaic": 10, "mixup": 0.0,
    "box": 7.5, "cls": 0.5, "dfl": 1.5,
    "project": "runs/train",
    "experiment_name": "wastevision",
    "mlflow_tracking_uri": "mlruns",
}
baseline_weights = train(baseline_config)
print("Baseline weights:", baseline_weights)

In [ ]:
from src.models.evaluate import evaluate

baseline_results = evaluate(baseline_weights, processed_dir / "data.yaml", split="val", imgsz=640)
print("Baseline val mAP50:", baseline_results["overall"]["mAP50"])
print("Baseline val mAP50-95:", baseline_results["overall"]["mAP50-95"])

## 7. Architecture + hyperparameter comparison
Trains yolov8n / yolov8s / yolov8m with augmentation tuned for small,
cluttered litter objects (per `configs/config.yaml`'s reasoning: stronger
box loss weight since localization matters more than classification here,
`close_mosaic` to avoid hurting small-object localization late in
training). All runs land in the same MLflow experiment for direct
comparison — open `mlflow ui` (cell below) rather than trusting whichever
run printed last.

In [ ]:
sweep_results = {}

for model_name, epochs in [("yolov8n.pt", 100), ("yolov8s.pt", 100), ("yolov8m.pt", 80)]:
    config = dict(baseline_config)
    config.update({
        "model": model_name,
        "epochs": epochs,
        "batch": 16 if model_name != "yolov8m.pt" else 8,  # larger model, smaller batch to fit memory
        "degrees": 5.0,
        "mixup": 0.1,
        "box": 8.0,  # upweighted: litter items are frequently small in frame
        "experiment_name": "wastevision",
    })
    weights = train(config)
    results = evaluate(weights, processed_dir / "data.yaml", split="val", imgsz=640)
    sweep_results[model_name] = {"weights": weights, **results["overall"]}
    print(model_name, "-> mAP50-95:", results["overall"]["mAP50-95"])

best_model_name = max(sweep_results, key=lambda k: sweep_results[k]["mAP50-95"])
best_weights = sweep_results[best_model_name]["weights"]
print(f"\nBest: {best_model_name} ({sweep_results[best_model_name]['mAP50-95']:.4f} mAP50-95), improvement over baseline: "
      f"{sweep_results[best_model_name]['mAP50-95'] - baseline_results['overall']['mAP50-95']:+.4f}")

## 8. Browse all runs side by side in MLflow

In [ ]:
!pip install -q pyngrok
get_ipython().system_raw("mlflow ui --backend-store-uri mlruns --port 5000 &")
from pyngrok import ngrok
print(ngrok.connect(5000))

## 9. Tune the confidence threshold for this checkpoint
`configs/config.yaml` and `src/models/inference.py` default to a generic
0.25 confidence cutoff. This finds the threshold that actually maximizes
F1 for the winning checkpoint, on the val set (test stays untouched until
final reporting, to avoid threshold-fitting on the number that goes in
the case study).

In [ ]:
from ultralytics import YOLO
from src.models.evaluate import select_operating_threshold

model = YOLO(str(best_weights))
val_metrics = model.val(data=str(processed_dir / "data.yaml"), split="val", imgsz=640, verbose=False)
tuned_threshold, tuned_f1 = select_operating_threshold(val_metrics.box.px, val_metrics.box.f1_curve)
print(f"Tuned confidence threshold: {tuned_threshold:.3f} (val F1: {tuned_f1:.4f})")

## 10. Final evaluation on the held-out test split
Reports both the standard metrics and a TTA (test-time augmentation)
variant — TTA is slower at inference time (multiple augmented passes per
image), so it's evaluated here as a data point on whether it's worth
enabling in production, not enabled in `inference.py` by default.

In [ ]:
from src.models.evaluate import write_report

final_results = evaluate(best_weights, processed_dir / "data.yaml", split="test", imgsz=640)
report_path = write_report(final_results, Path("eval_results"))
print(open(report_path).read())

tta_metrics = model.val(data=str(processed_dir / "data.yaml"), split="test", imgsz=640, augment=True, verbose=False)
print(f"\nTTA mAP50: {tta_metrics.box.map50:.4f}  (no-TTA: {final_results['overall']['mAP50']:.4f})")
print("If TTA's gain is small, it's not worth the extra inference latency for a live API.")

## 11. Error analysis
Starting point for `CASE_STUDY.md` section 8: pull predictions on the
worst-performing class (top row of the report above) and inspect them for
patterns — visual confusion with a similar material, small-object misses,
or TACO labeling ambiguity.

In [ ]:
import glob, random
from src.models.inference import load_inference_service

worst_class = min(
    (k for k, v in final_results["per_class"].items() if v["AP50"] is not None),
    key=lambda k: final_results["per_class"][k]["AP50"],
    default=None,
)
print("Worst-performing class:", worst_class, final_results["per_class"].get(worst_class))

service = load_inference_service(str(best_weights), conf_threshold=tuned_threshold, device=0)
test_images = glob.glob(str(processed_dir / "test" / "images" / "*"))
for path in random.sample(test_images, min(10, len(test_images))):
    detections = service.predict(path)
    print(Path(path).name, "->", [(d.material_group, round(d.confidence, 2)) for d in detections])

## 12. Export to ONNX + benchmark CPU latency
`DEPLOYMENT.md` Option A/B/C all serve on CPU (free/cheap tiers). This
exports the winning checkpoint to ONNX and measures per-image latency so
you know before deploying whether the free CPU tier is actually viable,
rather than finding out after `docker compose up`.

In [ ]:
import time

onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
print("Exported:", onnx_path)

onnx_model = YOLO(onnx_path)
sample_images = test_images[:10]
onnx_model.predict(sample_images[0], device="cpu", verbose=False)  # warm up; skip cold-start cost in the timing

start = time.time()
for path in sample_images:
    onnx_model.predict(path, device="cpu", verbose=False)
elapsed_ms = (time.time() - start) / len(sample_images) * 1000
print(f"Mean CPU latency: {elapsed_ms:.1f} ms/image ({1000/elapsed_ms:.1f} images/sec)")

## 13. Deployment sanity check
Runs the *exact* `InferenceService` the FastAPI route and Gradio app call
— not a reimplementation — against the exported ONNX weights, on a real
test image, before anything touches Docker.

In [ ]:
from src.models.inference import load_inference_service

deploy_service = load_inference_service(str(onnx_path), conf_threshold=tuned_threshold, device="cpu")
sample_detections = deploy_service.predict(sample_images[0])
print(f"{len(sample_detections)} detection(s) on {Path(sample_images[0]).name}:")
for d in sample_detections:
    print(" -", d.to_dict())

assert all(0.0 <= d.confidence <= 1.0 for d in sample_detections)
assert all(d.material_group in MATERIAL_GROUPS for d in sample_detections)
print("\nOK — ready to copy weights into models/ and build the Docker image.")

## 14. Save artifacts
Colab's disk is ephemeral. Save both the PyTorch checkpoint (for
retraining/fine-tuning later) and the ONNX export (what `DEPLOYMENT.md`
actually serves) to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dest = Path("/content/drive/MyDrive/wastevision")
dest.mkdir(parents=True, exist_ok=True)
!cp {best_weights} {dest}/best.pt
!cp {onnx_path} {dest}/best.onnx
!cp eval_results/report.md {dest}/report.md
print("Saved to", dest)
print(f"\nFor local deployment: copy best.pt into your repo's models/ directory, "
      f"then `docker compose up` (see DEPLOYMENT.md). Recommended inference "
      f"confidence threshold for this checkpoint: {tuned_threshold:.3f}.")